# Auto building & field extraction from SAM masks — Douala, Cameroon

End-to-end walkthrough: pull a satellite basemap for a Douala AOI, run Segment Anything (SAM) automatic mask generation with `samgeo`, vectorise the masks, then drive the **same pure-numpy analytics core** used by the tests to count buildings and measure footprint / field areas.

**Inspired by and built on** [`opengeos/segment-geospatial`](https://github.com/opengeos/segment-geospatial). The contribution here is the downstream quantification, not the segmentation itself.

## 0. Setup (Colab / GPU)

SAM is heavy: it needs `torch` and a CUDA **GPU** to be practical, and the checkpoint is large. Run this notebook on **Google Colab with a GPU runtime** (Runtime -> Change runtime type -> GPU) or a local GPU box.

The pure-numpy analytics core (Section 5) needs none of that — only numpy.

In [ ]:
# On Colab, install the stack (skip if your env already has it):
# %pip install segment-geospatial leafmap rasterio geopandas shapely
#
# Confirm a GPU is visible (recommended for SAM):
import torch

print("CUDA available:", torch.cuda.is_available())

## 1. Configuration

All analysis choices come from `config/aoi.yaml`: the Douala bbox, basemap source/zoom, pixel size, and the building/field area thresholds.

In [ ]:
import yaml

with open("../config/aoi.yaml") as fh:
    cfg = yaml.safe_load(fh)

aoi = cfg["aoi"]["bbox"]
bbox = [aoi["min_lon"], aoi["min_lat"], aoi["max_lon"], aoi["max_lat"]]
pixel_size_m = cfg["analysis"]["pixel_size_m"]
bbox, pixel_size_m

## 2. Pull basemap imagery for the AOI

`samgeo.tms_to_geotiff` downloads the basemap tiles covering the bbox into a single GeoTIFF. Keep the bbox small (a few km) so the tile count and SAM runtime stay tractable.

In [ ]:
from samgeo import tms_to_geotiff

image_tif = "../outputs/douala_image.tif"
tms_to_geotiff(
    output=image_tif,
    bbox=bbox,
    zoom=cfg["basemap"]["zoom"],
    source=cfg["basemap"]["source"],
    overwrite=True,
)

## 3. Run SAM automatic mask generation

`SamGeo.generate` runs the SAM automatic mask generator over the image and writes an **integer-labelled** mask GeoTIFF (one integer per object). This is the heavy, GPU-bound step.

Equivalent one-liner from the CLI: `samgeo-post segment --config config/aoi.yaml`.

In [ ]:
from samgeo import SamGeo

mask_tif = "../outputs/douala_masks.tif"
sam = SamGeo(model_type="vit_h", sam_kwargs=None)
sam.generate(image_tif, output=mask_tif, foreground=True, unique=True)

## 4. Vectorise the masks

Polygonise the labelled raster into vector features with a per-polygon area, using the lazy `samgeo_post.vectorize` wrapper (rasterio + shapely + geopandas).

In [ ]:
import rasterio

from samgeo_post.vectorize import masks_to_geojson

with rasterio.open(mask_tif) as ds:
    labeled = ds.read(1)
    transform, crs = ds.transform, ds.crs

gdf = masks_to_geojson(
    labeled,
    transform=transform,
    crs=crs,
    pixel_size_m=pixel_size_m,
    out_path="../outputs/douala_features.geojson",
)
gdf.head()

## 5. Quantify with the pure-numpy core

This is the tested contribution. We feed the **same labelled array** into `samgeo_post.analytics`: region properties, then split into the building-size band and the oversized field remainder using the m² thresholds from the config, and total the areas. No heavy deps here — this is exactly what the demo and the unit tests exercise.

In [ ]:
from samgeo_post.analytics import (
    area_hectares,
    filter_by_area,
    pixels_to_area,
    region_props,
)

px_min = cfg["analysis"]["building_min_area_m2"] / pixel_size_m**2
px_max = cfg["analysis"]["building_max_area_m2"] / pixel_size_m**2
field_min_px = cfg["analysis"]["field_min_area_m2"] / pixel_size_m**2

props = region_props(labeled)
buildings = filter_by_area(props, min_px=int(px_min), max_px=int(px_max))
fields = filter_by_area(props, min_px=int(field_min_px))

building_areas = [pixels_to_area(p["area_px"], pixel_size_m) for p in buildings]
field_px = sum(p["area_px"] for p in fields)

summary = {
    "n_buildings": len(buildings),
    "mean_building_footprint_m2": (sum(building_areas) / len(buildings))
    if buildings
    else 0.0,
    "total_building_area_m2": sum(building_areas),
    "field_area_hectares": area_hectares(pixels_to_area(field_px, pixel_size_m)),
}
summary

## 6. Before / after maps

Show the basemap and the extracted footprints / fields side by side with `leafmap`, then drop the GeoJSON on top for inspection against the imagery.

In [ ]:
import leafmap

m = leafmap.Map(center=[(bbox[1] + bbox[3]) / 2, (bbox[0] + bbox[2]) / 2], zoom=16)
m.add_raster(image_tif, layer_name="Basemap")
m.add_vector("../outputs/douala_features.geojson", layer_name="Extracted features")
m

## 7. Limitations and validation

- **SAM over-segments / under-segments.** Adjacent buildings can merge, and shadows or roof texture can split one roof; the area band filters some of this but not all.
- **Basemap, not a true ortho.** Tile imagery has lean and seam artefacts; footprints are approximate and the pixel size is nominal.
- **No per-class labels.** SAM is class-agnostic; “building” vs “field” here is inferred purely from the area band, not a classifier.
- **Validate before trusting counts.** Compare the extracted footprints against OpenStreetMap buildings or a cadastre for the same AOI, using `mask_iou` and footprint-count agreement, before quoting numbers.